# Ex14b RNN stock return predictions

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/master/tf2.0/sbux.csv')
df.head()

In [ ]:
df['prevClose'] = df['close'].shift(1)


In [ ]:
df.head()

In [ ]:
D = 1
T = 20
K = 1
M = 5
L = 1
bi = False
dropout = 0
lr = 0.01

### we're interested in stock return which defined by this formula $R :=\frac{V_{close}-V_{prev_close}}{V_{prev_close}}$

In [ ]:
df['return'] = (df['close']-df['prevClose'])/df['prevClose']

In [ ]:
df.head()

In [ ]:
plt.plot(df['return'])
plt.title('return values')
plt.show()

let's see how are return scatter along the histogram

In [ ]:
df['return'].hist()

values are extremely small and centered around 0.00

In [ ]:
series = df['return'].values[1:].reshape(-1,1)

In [ ]:
scaler=StandardScaler()
N = len(series)

scaler.fit(series[:N//2])
series = scaler.transform(series).flatten()

In [ ]:
X,y =[],[]
for t in range(N- T):
    X.append(series[t:t+T])
    y.append(series[t+T])

In [ ]:
N = len(X)
print(N)


In [ ]:
X = np.array(X).reshape(-1,T,D).astype(np.float32)
y = np.array(y).reshape(-1,1).astype(np.float32)

X_train,X_test =np.array(X[:N//2]),np.array(X[N//2:])
y_train,y_test = y[:N//2],y[N//2:]


In [ ]:
device = print('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
X_train = torch.from_numpy(X_train).to(device)
X_test = torch.from_numpy(X_test).to(device)
y_train = torch.from_numpy(y_train).to(device)
y_test = torch.from_numpy(y_test).to(device)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size,hidden_size,output_size,num_layers,bi=True,dropout=0.2):
        super().__init__()
        self.D =input_size
        self.L = num_layers
        self.M = hidden_size
        self.K = output_size

        self.biFactor = 2 if bi else 1

        self.rnn = nn.LSTM(input_size=self.D,
                           hidden_size=self.M,
                           num_layers=self.L,
                           batch_first=True,
                           bidirectional=bi,dropout=dropout)
        self.fc = nn.Linear(self.M * self.biFactor, self.K)

    def forward(self,X):
        h0 = torch.zeros(self.biFactor*self.L,X.size(0),self.M).to(device)
        c0 = torch.zeros(self.biFactor*self.L,X.size(0),self.M).to(device)

        out,(hn,cn) = self.rnn(X)
        out = self.fc(out[:,-1,:])
        return out


In [ ]:
epochs = 200

In [ ]:
lstm = LSTMModel(D,M,K,L,bi=bi,dropout=dropout).to(device)
optimizer = torch.optim.Adam(lstm.parameters(),lr=lr)
criterion = nn.MSELoss().to(device)
# scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer,max_lr=1e-3,steps_per_epoch=len(X_train), epochs=epochs)

In [ ]:
def single_epoch(model,optimizer: torch.optim.Optimizer,criterion: nn.Module,X,y,is_training=True):
    optimizer.zero_grad()

    input = X.view(-1,T,D).to(device)
    y_hat = model(input)
    loss = criterion(y_hat,y)

    if is_training:
        loss.backward()
        optimizer.step()
        # scheduler.step(loss)
    
    return y_hat,loss.item()

    

In [ ]:
train_losses = []
test_losses = []

for epoch in range(epochs):
    optimizer.zero_grad()
    _,train_loss = single_epoch(lstm,optimizer,criterion,X_train,y_train,True)
    _,test_loss = single_epoch(lstm,optimizer,criterion,X_test,y_test,False)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    print(f'{epoch}/{epochs} train loss {train_losses[-1]} test loss {test_losses[-1]}')

In [ ]:

print(f'lstm(D={D},M={M},K={K},L={L},bidirectional = {bi} dropout={dropout}) lr={lr}')

<b>scheduler was ReduceOnPlatue(optimizer,'min',patience=10) criterion = nn.MSE()</b>

lstm(D=1,M=5,K=1,L=1,bidirectional = False) 799/800 train loss 0.6069036722183228 test loss 2.007629156112671


lstm(D=1,M=5,K=1,L=5,bidirectional = False) 799/800 train loss 1.0541598796844482 test loss 1.0733938217163086

lstm(D=1,M=32,K=1,L=5,bidirectional = False) 799/800 train loss 0.010844520293176174 test loss 3.120149612426758

lstm(D=1,M=32,K=1,L=16,bidirectional = False) 799/800 train loss 1.0540679693222046 test loss 1.0718467235565186

lstm(D=1,M=16,K=1,L=5,bidirectional = False dropout=0.2) lr=0.001 799/800 train loss 1.054933786392212 test loss 1.0708281993865967

<b>scheduler was OneCycleLR(optimizer,max_lr=1e-3,steps_per_epoch=len(X_train), epochs=epochs) criterion = nn.SmoothL1Loss(beta=1.0)</b>
lstm(D=1,M=16,K=1,L=5,bidirectional = False dropout=0.25) lr=0.0005 799/800 train loss 0.4005145728588104 test loss 0.38926345109939575 


In [ ]:
plt.plot(train_losses,color='orange',label='train')
plt.plot(test_losses,color='red',label='test')
plt.legend()
plt.show()

In [ ]:
y_hats = []
X_last =X_test[0]

for i in range(y_test.shape[0]):
    y_hat,_ = single_epoch(lstm,optimizer,criterion,X_last,y_test[i],False)
    y_hats.append(y_hat.item())
    print('pre',X_last)
    X_last = torch.cat((X_last[1:],y_hat))
    print('post',X_last)

In [ ]:
ys = y_test.detach().cpu().numpy()

In [ ]:
plt.plot(y_hats,color='blue',label='y_hat')
plt.plot(ys,color='green',label='y')
plt.legend()
plt.show()

In [ ]:
inps = X_test.view(-1,T,D).to(device)
y_hats = lstm(inps)
y_hats = y_hats.detach().cpu().numpy()

In [ ]:
plt.plot(y_test,label='y_true',color='green')
plt.plot(y_hats,label='y_hat',color='blue')

plt.legend()
plt.show()